# 图片召回离线实验

**目标**: 评测不同"字段组合策略"下的图片召回效果,选出最优策略。

**数据假设**(详见 spec):
- 数据根 `DATA_ROOT` 下有 `images/`(图片,按随机串子目录)与 `output/`(每文档一个 json)。
- 每个 json 的 `figures[]` 中,每个元素有 `image_link`、`search_queries`、以及 3 个图片理解字段。
- 一条 query 的正确答案 = 它所属的那张图;top-K 召回的图里出现该图即命中。

**运行顺序**: 从上到下依次执行各 cell。
- **Cell 1 CONFIG** 是唯一需要你改的地方(数据路径、模型路径、字段名、策略)。
- **Cell 9** 跑全策略对比;**Cell 10** 交互看召回图片。

**评测口径**: 检索 top-N chunk → 折叠回图(同图取最靠前名次)→ 在图排名上算 Recall@K / MRR。
这样每图 chunk 数不同的策略(1 vs 3)也能公平对比。


In [ ]:
# ============================================================
# Cell 1: CONFIG —— 唯一需要你修改的地方 👈
# ============================================================
from pathlib import Path

# 数据根目录: 内含 images/ 和 output/ 两个子目录
# 👈 改成你服务器上的真实路径
DATA_ROOT = Path("./data")

# embedding 模型路径: HuggingFace id 或本地目录均可(768 维, 约 0.3b)
# 👈 改成你服务器上的模型路径/名称
MODEL_PATH = "./models/gte-model"

# 三个图片理解字段在 figures[] 元素里的真实 key 名
# 👈 等全量数据到位后,改成真实字段名
FIELD_A = "field_a"
FIELD_B = "field_b"
FIELD_C = "field_c"
# 字段代号(A/B/C) -> 真实 key。策略里用代号,便于阅读。
FIELD_KEYS = {"A": FIELD_A, "B": FIELD_B, "C": FIELD_C}

# 字段组合策略: 名称 -> 分组(每个子列表是一个 chunk 的字段代号组合)
# 👈 可增删策略;代号 A/B/C 对应上面 FIELD_KEYS
STRATEGIES = {
    "all_in_one":   [["A", "B", "C"]],          # 三字段拼成一个 chunk
    "AB_C":         [["A", "B"], ["C"]],         # A+B 一个 chunk, C 独立
    "AC_B":         [["A", "C"], ["B"]],
    "BC_A":         [["B", "C"], ["A"]],
    "all_separate": [["A"], ["B"], ["C"]],       # 三字段各自独立
}

# 组内字段拼接分隔符
SEP = " | "

# 评测的 K 值
K_LIST = [1, 3, 5, 10, 20]

# 检索时每条 query 取的 chunk 数 = max(K_LIST) * TOPN_FACTOR。
# 因为要把 chunk 折叠回图,需留余量保证折叠后仍有 >= max(K) 张不同的图。
# 👈 若某策略每图 chunk 多导致折叠后图不够,调大这个系数。
TOPN_FACTOR = 5

# 推理 batch size 👈 显存不够就调小
BATCH_SIZE = 64

# 设备: "auto" 自动选 GPU/CPU;也可写 "cuda" / "cpu"
DEVICE = "auto"

# embedding 缓存目录(.npy 落盘,重跑复用)
CACHE_DIR = Path("./cache_emb")
CACHE_DIR.mkdir(exist_ok=True)

# 数据子目录名(用于路径相对化锚点 & 拼接)
IMAGES_DIRNAME = "images"
OUTPUT_DIRNAME = "output"

print("CONFIG loaded. DATA_ROOT =", DATA_ROOT.resolve())
print("策略数:", len(STRATEGIES), "| K_LIST:", K_LIST)


In [ ]:
# ============================================================
# Cell 2: 环境 & 工具函数(imports / 设备检测 / 计时)
# ============================================================
import os, json, time, glob
import numpy as np
import pandas as pd

def resolve_device(device_cfg):
    """根据 CONFIG 的 DEVICE 决定实际设备。auto: 有 CUDA 用 cuda,否则 cpu。"""
    if device_cfg != "auto":
        return device_cfg
    try:
        import torch
        return "cuda" if torch.cuda.is_available() else "cpu"
    except Exception:
        return "cpu"

DEVICE_RESOLVED = resolve_device(DEVICE)
print("使用设备:", DEVICE_RESOLVED)

class Timer:
    """简易计时上下文管理器: with Timer('编码'): ..."""
    def __init__(self, name):
        self.name = name
    def __enter__(self):
        self.t = time.time()
        return self
    def __exit__(self, *a):
        print("[{}] 用时 {:.2f}s".format(self.name, time.time() - self.t))


In [ ]:
# ============================================================
# 核心纯逻辑函数(路径处理 / chunk 生成 / 召回指标 / 数据解析)
# 这些函数无 I/O、无模型依赖,可被单元测试覆盖。
# ============================================================

def relativize_image_link(image_link, images_dirname="images"):
    """把图片绝对路径转成相对数据根的路径。
    锚点 = 路径中最后一个名为 images_dirname 的目录片段;取其(含该片段)之后的部分。
    若找不到锚点,回退为纯文件名(由调用方计入异常计数)。
    👈 可在此修改: 若你的图片根目录名不是 'images',改 images_dirname(或在 CONFIG 改)。
    """
    norm = image_link.replace(chr(92), "/")  # chr(92) 是反斜杠,兼容 Windows 路径
    parts = norm.split("/")
    idx = None
    for i in range(len(parts) - 1, -1, -1):   # 从右往左找最后一个 images_dirname
        if parts[i] == images_dirname:
            idx = i
            break
    if idx is None:
        return parts[-1]                       # 回退: 文件名
    return "/".join(parts[idx:])


def build_chunks_for_image(image_id, field_values, groups, sep=" | "):
    """按策略 groups 把一张图的字段切成若干 chunk。
    field_values: {字段代号: 字段值},例如 {"A": "...", "B": "...", "C": "..."}
    groups:       [[字段代号, ...], ...],每个子列表是一个 chunk 的字段组合
    返回:         [(chunk_id, image_id, chunk_text), ...]
    规则: 空字段跳过;整组全空则不产出该 chunk。
    👈 可在此修改: sep 是组内字段拼接分隔符(也可在 CONFIG 改 SEP)。
    """
    chunks = []
    for gi, group in enumerate(groups):
        vals = []
        for key in group:
            v = field_values.get(key)
            if v is None:
                continue
            v = str(v).strip()
            if v == "":
                continue
            vals.append(v)
        if not vals:
            continue                            # 整组全空,跳过
        text = sep.join(vals)
        chunk_id = "{}__g{}".format(image_id, gi)
        chunks.append((chunk_id, image_id, text))
    return chunks


def fold_chunks_to_images(ranked_image_ids):
    """把"按相似度降序的 chunk 的 image_id 列表"折叠成去重保序的图排名。
    同一张图多次出现,只保留最靠前那次(第一次出现)。
    这是评测口径的关键: 让 chunk 数不同的策略可以公平对比。
    """
    seen = set()
    folded = []
    for iid in ranked_image_ids:
        if iid in seen:
            continue
        seen.add(iid)
        folded.append(iid)
    return folded


def gold_rank(folded_image_ids, gold_image_id):
    """返回 gold 图在折叠后图排名中的 1-based 名次;未命中返回 None。"""
    for i, iid in enumerate(folded_image_ids):
        if iid == gold_image_id:
            return i + 1
    return None


def recall_at_k(ranks, k):
    """Recall@K: 命中名次 <= K 的 query 占比。ranks 为每条 query 的名次(None=未命中)。"""
    if not ranks:
        return 0.0
    hit = sum(1 for r in ranks if r is not None and r <= k)
    return hit / len(ranks)


def mrr(ranks):
    """MRR(平均倒数名次)。未命中贡献 0。"""
    if not ranks:
        return 0.0
    total = 0.0
    for r in ranks:
        if r is not None:
            total += 1.0 / r
    return total / len(ranks)


def parse_figures_from_doc(doc, doc_id, field_keys, images_dirname="images"):
    """解析单个 output json(已 load 成 dict)的 figures[]。
    field_keys: {"A": 真实key, "B": 真实key, "C": 真实key}
    返回 (rows, stats):
      rows : [{doc_id, image_id, rel_path, fields:{A,B,C}, search_queries:[...]}, ...]
      stats: {n_images, n_queries, n_path_fallback}
    image_id 用相对路径(全局唯一标识一张图)。
    """
    rows = []
    n_queries = 0
    n_path_fallback = 0
    for fig in doc.get("figures", []):
        link = fig.get("image_link", "")
        rel = relativize_image_link(link, images_dirname)
        if "/" not in rel:                      # 回退成纯文件名 => 没找到锚点
            n_path_fallback += 1
        fields = {}
        for code, real_key in field_keys.items():
            v = fig.get(real_key)
            fields[code] = "" if v is None else str(v)
        sq = fig.get("search_queries") or []
        if not isinstance(sq, list):
            sq = []
        n_queries += len(sq)
        rows.append({
            "doc_id": doc_id,
            "image_id": rel,
            "rel_path": rel,
            "fields": fields,
            "search_queries": sq,
        })
    stats = {"n_images": len(rows), "n_queries": n_queries, "n_path_fallback": n_path_fallback}
    return rows, stats


In [ ]:
# ============================================================
# Cell 3: 数据加载 & 解析 —— 遍历 output/*.json,产出"每图一行"表
# ============================================================
output_dir = DATA_ROOT / OUTPUT_DIRNAME
json_files = sorted(glob.glob(str(output_dir / "*.json")))
print("找到 json 文档数:", len(json_files))

all_rows = []
total_stats = {"n_images": 0, "n_queries": 0, "n_path_fallback": 0}
for jf in json_files:
    doc_id = os.path.splitext(os.path.basename(jf))[0]
    try:
        with open(jf, "r", encoding="utf-8") as fh:
            doc = json.load(fh)
    except Exception as e:
        print("跳过无法解析的 json:", jf, e)
        continue
    rows, stats = parse_figures_from_doc(doc, doc_id, FIELD_KEYS, IMAGES_DIRNAME)
    all_rows.extend(rows)
    for k in total_stats:
        total_stats[k] += stats[k]

# 每图一行的 DataFrame(fields 展开成 A/B/C 三列,便于查看)
images_df = pd.DataFrame([{
    "image_id": r["image_id"],
    "doc_id": r["doc_id"],
    "rel_path": r["rel_path"],
    "A": r["fields"]["A"],
    "B": r["fields"]["B"],
    "C": r["fields"]["C"],
    "n_queries": len(r["search_queries"]),
    "search_queries": r["search_queries"],
} for r in all_rows])

print("图片总数:", total_stats["n_images"])
print("query 总数:", total_stats["n_queries"])
print("路径相对化失败(回退文件名)数:", total_stats["n_path_fallback"])
# image_id 唯一性检查 👈 若有重复说明文件名跨目录撞了,需换 image_id 方案
dup = images_df["image_id"].duplicated().sum() if len(images_df) else 0
print("重复 image_id 数:", dup, "(应为 0)")
images_df.head()


In [ ]:
# ============================================================
# Cell 4: 构建评测 query 集 —— 每行 = (query, gold_image_id)
# ============================================================
query_records = []
for r in all_rows:
    for q in r["search_queries"]:
        q = (q or "").strip()
        if q == "":
            continue
        query_records.append({"query": q, "gold_image_id": r["image_id"]})

queries_df = pd.DataFrame(query_records)
print("有效 query 数:", len(queries_df))
if len(queries_df):
    print("覆盖的 gold 图数:", queries_df["gold_image_id"].nunique())
    avg = len(queries_df) / queries_df["gold_image_id"].nunique()
    print("每图平均 query 数: {:.2f}".format(avg))
queries_df.head()


In [ ]:
# ============================================================
# Cell 5: chunk 生成器 —— 给定策略名,产出该策略下全体图的 chunk 表
# ============================================================
def build_chunks_for_strategy(strategy_name):
    """对所有图应用某策略,返回 DataFrame[chunk_id, image_id, text]。"""
    groups = STRATEGIES[strategy_name]
    rows = []
    for r in all_rows:
        rows.extend(build_chunks_for_image(r["image_id"], r["fields"], groups, sep=SEP))
    return pd.DataFrame(rows, columns=["chunk_id", "image_id", "text"])

# 预览: 看每个策略产出的 chunk 数
for sname in STRATEGIES:
    cdf = build_chunks_for_strategy(sname)
    print("策略 {:14s} chunk 数: {}".format(sname, len(cdf)))

# 预览第一个策略的前几条
_first = list(STRATEGIES.keys())[0]
print("\n策略 [{}] 样例:".format(_first))
build_chunks_for_strategy(_first).head()


In [ ]:
# ============================================================
# Cell 6: Embedding 模型加载 & 批量编码(带 .npy 缓存)
# ============================================================
from tqdm.auto import tqdm

# 全局只加载一次模型 👈 用 sentence-transformers
_MODEL = None
def get_model():
    global _MODEL
    if _MODEL is None:
        from sentence_transformers import SentenceTransformer
        print("加载模型:", MODEL_PATH, "-> 设备", DEVICE_RESOLVED)
        _MODEL = SentenceTransformer(str(MODEL_PATH), device=DEVICE_RESOLVED,
                                     trust_remote_code=True)
    return _MODEL

def encode_texts(texts):
    """批量编码并 L2 归一化,返回 float32 ndarray [N, dim]。"""
    model = get_model()
    embs = model.encode(
        list(texts),
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,    # 归一化 -> 内积 = 余弦
    )
    return embs.astype("float32")

def encode_with_cache(texts, cache_name):
    """带缓存的编码。cache_name 决定 .npy 文件名。
    👈 注意: 改了字段名/分隔符/模型后,请删除 cache_emb/ 下旧缓存重算。
    """
    cache_file = CACHE_DIR / (cache_name + ".npy")
    if cache_file.exists():
        print("命中缓存:", cache_file)
        return np.load(cache_file)
    embs = encode_texts(texts)
    np.save(cache_file, embs)
    print("写入缓存:", cache_file, embs.shape)
    return embs

print("Embedding 单元就绪。(模型在首次调用 get_model() 时才加载)")


In [ ]:
# ============================================================
# Cell 7: FAISS 建索引 + 检索(返回每条 query 的 top-N chunk 的 image_id 序列)
# ============================================================
import faiss

def build_index(chunk_embs):
    """用归一化向量建内积索引(= 余弦相似度)。"""
    dim = chunk_embs.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(chunk_embs)
    return index

def retrieve_ranked_image_ids(query_embs, index, chunk_image_ids, topn):
    """检索 top-N chunk,返回每条 query 的 [image_id, ...](按相似度降序,未折叠)。
    chunk_image_ids: 与索引中向量顺序一致的 image_id 列表。
    """
    topn = min(topn, len(chunk_image_ids))
    scores, idxs = index.search(query_embs, topn)
    results = []
    for row in idxs:
        results.append([chunk_image_ids[i] for i in row if i != -1])
    return results, scores, idxs

def run_strategy_retrieval(strategy_name, query_embs):
    """对某策略: 生成 chunk -> 编码(缓存)-> 建索引 -> 检索。
    返回 (chunk_df, ranked_image_ids_per_query, scores, idxs)。
    """
    chunk_df = build_chunks_for_strategy(strategy_name)
    chunk_embs = encode_with_cache(chunk_df["text"].tolist(),
                                   cache_name="chunks__" + strategy_name)
    index = build_index(chunk_embs)
    topn = max(K_LIST) * TOPN_FACTOR
    ranked, scores, idxs = retrieve_ranked_image_ids(
        query_embs, index, chunk_df["image_id"].tolist(), topn)
    return chunk_df, ranked, scores, idxs

print("检索单元就绪。")


In [ ]:
# ============================================================
# Cell 8: 单策略指标 —— 折叠回图 + Recall@K / MRR / 命中名次分布
# ============================================================
def compute_ranks(ranked_image_ids_per_query, gold_image_ids):
    """对每条 query: 折叠 chunk->图,求 gold 图名次(None=未命中)。返回 ranks 列表。"""
    ranks = []
    for ranked, gold in zip(ranked_image_ids_per_query, gold_image_ids):
        folded = fold_chunks_to_images(ranked)
        ranks.append(gold_rank(folded, gold))
    return ranks

def metrics_from_ranks(ranks):
    """返回 {recall@k..., mrr, hit_rate, n_queries}。"""
    out = {}
    for k in K_LIST:
        out["recall@{}".format(k)] = recall_at_k(ranks, k)
    out["mrr"] = mrr(ranks)
    out["hit_rate"] = (sum(1 for r in ranks if r is not None) / len(ranks)) if ranks else 0.0
    out["n_queries"] = len(ranks)
    return out

def evaluate_strategy(strategy_name, query_embs, gold_image_ids):
    """端到端评测一个策略,返回 (metrics_dict, ranks)。"""
    _, ranked, _, _ = run_strategy_retrieval(strategy_name, query_embs)
    ranks = compute_ranks(ranked, gold_image_ids)
    return metrics_from_ranks(ranks), ranks

print("指标单元就绪。")


In [ ]:
# ============================================================
# Cell 9: 全策略评测对比(query 只编码一次,复用到所有策略)
# ============================================================
import matplotlib.pyplot as plt

# query 编码一次(带缓存)
gold_ids = queries_df["gold_image_id"].tolist()
query_embs = encode_with_cache(queries_df["query"].tolist(), cache_name="queries")

# 逐策略评测
results_rows = []
ranks_per_strategy = {}
for sname in STRATEGIES:
    with Timer("评测 " + sname):
        m, ranks = evaluate_strategy(sname, query_embs, gold_ids)
    results_rows.append({"strategy": sname, **m})
    ranks_per_strategy[sname] = ranks
    print(sname, "->", {k: round(v, 4) for k, v in m.items()
                        if k.startswith("recall") or k == "mrr"})

compare_df = pd.DataFrame(results_rows).set_index("strategy")
print("\n=== 策略对比汇总表 ===")
try:
    display(compare_df)
except NameError:
    print(compare_df)

# 存 CSV 👈 结果落盘
compare_df.to_csv("strategy_comparison.csv")
print("已保存 strategy_comparison.csv")

# 柱状图: 各策略 Recall@K
recall_cols = ["recall@{}".format(k) for k in K_LIST]
ax = compare_df[recall_cols].plot(kind="bar", figsize=(10, 5))
ax.set_title("各策略 Recall@K 对比"); ax.set_ylabel("Recall"); ax.set_ylim(0, 1)
plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.show()

# 命中名次分布直方图(各策略叠加)
plt.figure(figsize=(10, 5))
for sname, ranks in ranks_per_strategy.items():
    hit_ranks = [r for r in ranks if r is not None]
    plt.hist(hit_ranks, bins=range(1, max(K_LIST) + 2), alpha=0.4, label=sname)
plt.title("命中名次分布(仅命中的 query)"); plt.xlabel("gold 图名次"); plt.ylabel("query 数")
plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
# ============================================================
# Cell 10: 交互召回 demo —— 看某条 query 的 top-K 召回图片(绿框=正确图)
# ============================================================
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image

# 选用于 demo 的策略 👈 改成你想看的策略名
DEMO_STRATEGY = list(STRATEGIES.keys())[0]
DEMO_TOPK = 5  # 👈 展示几张

# 预备该策略的检索资产(只算一次)
_demo_chunk_df = build_chunks_for_strategy(DEMO_STRATEGY)
_demo_chunk_embs = encode_with_cache(_demo_chunk_df["text"].tolist(),
                                     cache_name="chunks__" + DEMO_STRATEGY)
_demo_index = build_index(_demo_chunk_embs)
_demo_chunk_image_ids = _demo_chunk_df["image_id"].tolist()
# image_id -> rel_path 映射(展示用)
_id2path = {r["image_id"]: r["rel_path"] for r in all_rows}

def demo_query(query_text=None, gold_image_id=None, topk=DEMO_TOPK):
    """展示一条 query 的 top-K 召回图片。query_text 为空则从评测集随机抽一条。"""
    if query_text is None:
        rec = queries_df.sample(1).iloc[0]
        query_text = rec["query"]; gold_image_id = rec["gold_image_id"]
    print("Query:", query_text)
    if gold_image_id is not None:
        print("正确图:", gold_image_id)

    q_emb = encode_texts([query_text])
    topn = topk * TOPN_FACTOR
    ranked, _, _ = retrieve_ranked_image_ids(q_emb, _demo_index, _demo_chunk_image_ids, topn)
    folded = fold_chunks_to_images(ranked[0])[:topk]

    fig, axes = plt.subplots(1, len(folded), figsize=(4 * len(folded), 4))
    if len(folded) == 1:
        axes = [axes]
    for ax, iid in zip(axes, folded):
        rel = _id2path.get(iid, iid)
        img_path = DATA_ROOT / rel
        try:
            ax.imshow(Image.open(img_path))
        except Exception:
            ax.text(0.5, 0.5, "无法读图\n" + str(rel), ha="center", va="center")
        hit = (gold_image_id is not None and iid == gold_image_id)
        ax.set_title(("[命中] " if hit else "") + rel.split("/")[-1], fontsize=9,
                     color=("green" if hit else "black"))
        ax.axis("off")
        if hit:  # 绿框高亮正确图
            ax.add_patch(Rectangle((0, 0), 1, 1, transform=ax.transAxes,
                                   fill=False, edgecolor="lime", linewidth=5))
    plt.tight_layout(); plt.show()

# 随机抽一条试跑 👈 也可手动指定: demo_query("你的查询词")
demo_query()
